# Face Mask Detection — Image Processing Project
Trains a mask/no-mask classifier by fine-tuning a pretrained CNN (MobileNet) on real photos.

**Before running:** unzip `converted_keras.zip` once so `keras_model.h5` and `labels.txt` exist directly
on disk, and set the 4 paths below to match your folders.

In [4]:
MODEL_PATH = "model/keras_model.h5"
LABELS_PATH = "model/labels.txt"
WITH_MASK_DIR = "images/with mask"
WITHOUT_MASK_DIR = "images/without mask" 

## 1. Import Libraries

In [5]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import glob, warnings
import numpy as np
import tensorflow as tf
import tf_keras as keras
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tf_keras.layers import DepthwiseConv2D
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
np.random.seed(42)
tf.random.set_seed(42)

## 2. Load the Pretrained CNN Model

In [6]:
class DepthwiseConv2DFix(DepthwiseConv2D):
    def __init__(self, **kwargs):
        kwargs.pop("groups", None)   # old export uses an arg newer Keras versions reject
        super().__init__(**kwargs)

mask_model = keras.models.load_model(MODEL_PATH, compile=False,
                                      custom_objects={"DepthwiseConv2D": DepthwiseConv2DFix})
mask_labels = [l.split(" ", 1)[1] for l in open(LABELS_PATH).read().splitlines()]

print("Input shape: ", mask_model.input_shape)
print("Output shape:", mask_model.output_shape)
print("Classes:     ", mask_labels)

Input shape:  (None, 224, 224, 3)
Output shape: (None, 2)
Classes:      ['With Mask', 'Without Mask']


## 3. Prepare & Preprocess the Data

In [ ]:
with_paths = glob.glob(f"{WITH_MASK_DIR}/*")[:150]
without_paths = glob.glob(f"{WITHOUT_MASK_DIR}/*")[:150]
all_paths = with_paths + without_paths
y_all = np.array([0]*len(with_paths) + [1]*len(without_paths))   # 0=With Mask, 1=Without Mask

def load_images(paths):
    return np.array([
        (np.asarray(Image.open(p).convert("RGB").resize((224, 224))).astype(np.float32) / 127.5) - 1
        for p in paths
    ])

X = load_images(all_paths)
X_train, X_test, y_train, y_test = train_test_split(X, y_all, test_size=0.2, random_state=42, stratify=y_all)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 4. Train (Fine-Tune) the Model

In [ ]:
mask_model.layers[0].trainable = False   # keep the CNN feature extractor frozen
mask_model.layers[1].trainable = True    # train only the small classifier head

mask_model.compile(optimizer=keras.optimizers.Adam(1e-4),
                    loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history = mask_model.fit(X_train, y_train, validation_data=(X_test, y_test),
                          epochs=5, batch_size=16, verbose=0)
print("Train accuracy:", round(history.history["accuracy"][-1], 3))
print("Val accuracy:  ", round(history.history["val_accuracy"][-1], 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(history.history["accuracy"], label="Train")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy"); axes[0].legend()
axes[1].plot(history.history["loss"], label="Train")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Loss"); axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Evaluate

In [ ]:
preds = mask_model.predict(X_test, verbose=0).argmax(axis=1)
print("Test accuracy:", round(accuracy_score(y_test, preds), 3))
print(classification_report(y_test, preds, target_names=mask_labels))

cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=mask_labels, yticklabels=mask_labels)
plt.title("Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()

## 6. CNN Concept: What the Convolution Layers See
A quick look inside the CNN — the feature map an early convolution layer produces for one real photo.

In [ ]:
cnn_backbone = mask_model.layers[0].layers[0]                 # the MobileNet CNN inside the model
first_conv_layer = next(l for l in cnn_backbone.layers if isinstance(l, keras.layers.Conv2D))
feature_map_model = keras.Model(cnn_backbone.input, first_conv_layer.output)

sample_img = X_test[0:1]
feature_maps = feature_map_model.predict(sample_img, verbose=0)[0]

fig, axes = plt.subplots(1, 6, figsize=(12, 2))
axes[0].imshow((sample_img[0] + 1) / 2); axes[0].set_title("Input"); axes[0].axis("off")
for i in range(1, 6):
    axes[i].imshow(feature_maps[:, :, i], cmap="viridis"); axes[i].set_title(f"Filter {i}"); axes[i].axis("off")
plt.suptitle(f'Feature maps from "{first_conv_layer.name}" (first Conv2D layer)')
plt.tight_layout(); plt.show()

## 7. Save & Reload the Trained Model

In [ ]:
mask_model.save("mask_model_finetuned.keras")

loaded_model = keras.models.load_model(
    "mask_model_finetuned.keras", custom_objects={"DepthwiseConv2DFix": DepthwiseConv2DFix}
)
print("Reloaded model input shape:", loaded_model.input_shape)

## 8. Test on a New Image

In [ ]:
def predict_mask(image_path, model=loaded_model):
    img = Image.open(image_path).convert("RGB").resize((224, 224))
    arr = (np.asarray(img).astype(np.float32) / 127.5) - 1
    pred = model.predict(np.expand_dims(arr, 0), verbose=0)[0]
    idx = int(np.argmax(pred))
    return mask_labels[idx], round(float(pred[idx]) * 100, 2)

new_image = glob.glob(f"{WITHOUT_MASK_DIR}/*")[200]
label, confidence = predict_mask(new_image)
plt.imshow(Image.open(new_image)); plt.axis("off")
plt.title(f"Prediction: {label} ({confidence}%)"); plt.show()